# Week 6 — Integrated Data Science Project
## Breast Cancer Diagnostic Intelligence System

This notebook guides the full data-science workflow: data collection, cleaning, EDA, statistical analysis, machine learning, evaluation and feature importance.

**Educational use only. This is not a clinical diagnostic tool.**

## 1. Imports and dataset collection

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loading import load_dataset
from src.preprocessing import clean_dataset

df = load_dataset()
print("Dataset shape:", df.shape)
display(df.head())

## 2. Data understanding and quality

In [ ]:
print(df.info())
display(df.describe().T)
print("Missing cells:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
display(df["diagnosis"].value_counts())

## 3. Data cleaning

In [ ]:
cleaned, quality_report = clean_dataset(df)
display(pd.Series(quality_report))
(ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
cleaned.to_csv(ROOT / "data" / "processed" / "wdbc_cleaned.csv", index=False)

## 4. Exploratory data analysis

In [ ]:
sns.countplot(data=cleaned, x="diagnosis", order=["B", "M"])
plt.title("Diagnosis Class Distribution")
plt.show()

sns.boxplot(data=cleaned, x="diagnosis", y="mean radius", order=["B", "M"])
plt.title("Mean Radius by Diagnosis")
plt.show()

plt.figure(figsize=(15, 12))
sns.heatmap(cleaned.drop(columns="diagnosis").corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

## 5. Statistical hypothesis testing

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu

malignant = cleaned.loc[cleaned.diagnosis == "M", "mean radius"]
benign = cleaned.loc[cleaned.diagnosis == "B", "mean radius"]

t_stat, p_value = ttest_ind(malignant, benign, equal_var=False)
u_stat, u_p = mannwhitneyu(malignant, benign, alternative="two-sided")

print("Malignant mean:", malignant.mean())
print("Benign mean:", benign.mean())
print("Welch t statistic:", t_stat)
print("Welch p-value:", p_value)
print("Mann–Whitney p-value:", u_p)
print("At alpha=0.05:", "Reject H0" if p_value < 0.05 else "Fail to reject H0")

## 6. Multicollinearity using VIF

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

Xv = cleaned.drop(columns="diagnosis").astype(float)
Xv = (Xv-Xv.mean()) / Xv.std(ddof=0).replace(0, 1)
Xv = Xv.replace([np.inf, -np.inf], np.nan).fillna(0)
Xv = sm.add_constant(Xv, has_constant="add")

vif = pd.DataFrame({
    "feature": [c for c in Xv.columns if c != "const"],
    "VIF": [variance_inflation_factor(Xv.values, i)
            for i, c in enumerate(Xv.columns) if c != "const"]
}).sort_values("VIF", ascending=False)
display(vif.head(15))

## 7. Machine learning and model comparison

In [ ]:
from sklearn.model_selection import train_test_split
from src.model_training import build_models
from src.model_evaluation import evaluate_models

X = cleaned.drop(columns="diagnosis")
y = (cleaned.diagnosis == "M").astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

results, best_name, best_model = evaluate_models(
    build_models(), X_train, X_test, y_train, y_test, ROOT / "reports"
)
print("Selected baseline model:", best_name)
display(results)

## 8. Feature importance

In [ ]:
from src.explainability import create_permutation_importance
importance = create_permutation_importance(
    best_model, X_test, y_test, ROOT / "visualizations"
)
display(importance.head(15))

## 9. Interpretation and conclusion

Write your interpretation using the real results above.

Discuss:
- Which model had the strongest cross-validated malignant recall?
- How did test precision, recall, F1 and ROC-AUC compare?
- Which features differed statistically between the two groups?
- What did VIF reveal about predictor redundancy?
- Which features were influential in permutation importance?
- What limitations apply to this dataset and experiment?

A hold-out test on this dataset is not external clinical validation. Do not describe this project as a clinically validated diagnostic system.